# Advanced Example: STL Geometry, Sampling, and Calibration Grid Generation

This tutorial builds on `introduction-tutorial.ipynb` and works through a more
realistic calibration workflow: **STL-based duct geometry** with multiple
material interfaces, a **dual-laser stage** configuration, and the
**inverse optimization** machinery needed to turn a set of desired 3D
calibration points into stage positions and G-code.

By the end you will have:

- Loaded a duct from STL files and built a multi-material optical system
  around it
- Positioned a dual-laser stage source and located an initial aim point
- Sampled the reachable volume and trained a fast estimator for the inverse
  problem (target point -> stage position)
- Used `Optimizer` to refine that estimate to micrometer precision
- Generated a small calibration grid with a travel-minimizing scan order
- Exported the result as G-code

If you have not yet gone through `introduction-tutorial.ipynb`, start there
first -- this notebook assumes you are already familiar with `OpticalSystem`,
`OpticalInterface`, `RayTracer`, and the surface-normal convention.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from hazy import Frame
from tqdm.notebook import tqdm

import laser_cross_calibration as lcc

## The Setup

The geometry is a straight octagonal duct: a fused-silica glass wall filled
with a water-glycerol mixture, viewed from outside through air. This mirrors
a real experimental use case -- calibrating cameras for flow measurements
inside a duct, where the beams refract at both the outer (air -> fused
silica) and inner (fused silica -> fluid) walls before crossing inside the
fluid.

The geometry is provided as two STL meshes, `simple-octagone-outer.stl` and
`simple-octagone-inner.stl` (`examples/stl-files/`), each describing one wall
of the duct. They were exported in millimeters, so we scale them to meters
(the framework's convention throughout) right after loading.

In [ ]:
root = Frame(name="root")

outer_wall = lcc.surfaces.TriSurface.from_stl_file(
    stl_path="stl-files/simple-octagone-outer.stl",
    is_smooth=False,
    frame=root,
    info="outer wall",
).scale(1e-3, 1e-3, 1e-3)

inner_wall = lcc.surfaces.TriSurface.from_stl_file(
    stl_path="stl-files/simple-octagone-inner.stl",
    is_smooth=False,
    frame=root,
    info="inner wall",
).scale(1e-3, 1e-3, 1e-3)

# air -> fused silica glass at the outer surface, fused silica glass -> fluid at the inner surface
outer_interface = lcc.tracing.OpticalInterface(
    geometry=outer_wall,
    material_pre=lcc.materials.AIR,
    material_post=lcc.materials.GLASS_FUSED_SILICA,
)
inner_interface = lcc.tracing.OpticalInterface(
    geometry=inner_wall,
    material_pre=lcc.materials.GLASS_FUSED_SILICA,
    material_post=lcc.materials.WATER_GLYCEROL_MIXTURE_90,
)

system = lcc.tracing.OpticalSystem(final_propagation_distance=1.0)
system.add_interface(outer_interface)
system.add_interface(inner_interface)

`is_smooth=False` keeps flat per-triangle normals rather than interpolating
them across vertices. An octagonal duct is already flat-faced by
construction -- each of its eight sides is a real, physically flat surface,
not a coarse approximation of something curved -- so there is nothing to
gain from smooth interpolation here. It matters more once you switch to
genuinely curved STL geometry (a rounded pipe bend, for example): a coarse
smooth-shaded mesh can meaningfully change the traced result, while a coarse
flat-shaded one just make the facets more visible.

Both meshes were already exported in the stage's native convention (XY as
the horizontal plane, Z vertical), so no additional rotation is needed here.
That is not true in general: when working from your own CAD exports, check
the geometry's orientation before tracing (see the surface-normal check in
the project README), the same way you would validate normal directions in
Blender.

In [ ]:
scene = lcc.visualization.Scene()
scene.bounds_min = (-0.35, -0.45, -0.25)
scene.bounds_max = (0.15, 0.05, 0.25)
scene.add_system(system)
scene.make_figure()

## The Laser Stage

The dual-laser stage has two configurable properties, independent of
whatever part is currently being measured:

- **`magic_angle`**: the angle between the stage and the duct. In practice
  this describes how far the arms of the stage are twisted relative to the
  part being measured. It's zero here for simplicity.
- **arm lengths**: the offset of each laser pointer from the stage's
  reference point, along its (rotated) X and Y axes respectively. They are
  equal here (0.25 m each) purely to keep this walkthrough simple -- on a
  real rig these are two independent, physically measured distances.

On a real setup these numbers describe the laser mount hardware itself, not
any specific optical system, so they carry over to any part you put in front
of the stage -- you measure them once and reuse them.

What is *not* fixed is where the stage needs to be positioned to actually
aim through **this** duct -- that depends on where the duct sits in the
world. In a real session you would place the part on the table, then jog the
stage until the beams roughly line up. Here, we get an approximate starting
position analytically: since both beam directions run exactly along the
stage's own (rotated) axes, the un-refracted crossing point of the two beams
is always at local coordinates `(arm_length_1, arm_length_2, 0)` relative to
the stage origin, regardless of translation. Solving that backwards for a
desired world-space aim point gives a good starting `stage_frame` placement.

In [ ]:
magic_angle_deg = 0
arm_length_1 = 0.25
arm_length_2 = 0.25

# axes of a rotated-only probe frame, used to solve for a starting
# translation below
probe_frame = root.make_child(name="probe").rotate_euler(
    z=magic_angle_deg, degrees=True
)
x_axis_global = np.array(probe_frame.x_axis.to_global())
y_axis_global = np.array(probe_frame.y_axis.to_global())

# rough aim point: somewhere inside the tube, ignoring refraction for now
approximate_aim_point = np.array([0.06, -0.135, 0.0])
starting_translation = approximate_aim_point - (
    arm_length_1 * x_axis_global + arm_length_2 * y_axis_global
)

stage_frame = root.make_child(name="stage").rotate_euler(
    z=magic_angle_deg, degrees=True
)
stage_frame.translate(
    x=starting_translation[0], y=starting_translation[1], z=starting_translation[2]
)

source = lcc.sources.DualLaserStageSource(
    origin=stage_frame.origin,
    arm1=stage_frame.x_axis * arm_length_1,
    arm2=stage_frame.y_axis * arm_length_2,
    direction1=stage_frame.y_axis,
    direction2=stage_frame.x_axis,
    display_scale=0.05,
)

In [ ]:
tracer = lcc.tracing.RayTracer(optical_system=system)
rays, intersections = tracer.trace_and_find_crossings(sources=[source], threshold=1e-3)

for i, ray in enumerate(rays):
    media_sequence = [medium.name for medium in ray.media_history]
    print(f"ray {i}: {' -> '.join(media_sequence)}")

print(f"\nfound {len(intersections)} crossing(s)")
if intersections:
    print("crossing point (world):", np.array(intersections[0].to_global()))

Both rays should show the full `air -> fused silica -> water-glycerol ->
fused silica -> ...` sequence: entering through the outer wall, crossing the
fluid, and exiting through the inner wall on the far side of the duct. If
you see `air` reappear right after the first `fused silica` entry, something
about the mesh or stage placement needs adjusting -- that sequence should be
clean for a beam that actually enters the duct.

In [ ]:
scene.clear_sources()
scene.add_source(source)
scene.clear_rays()
scene.add_rays(rays=rays)
if intersections:
    scene.clear_points()
    scene.add_point(*np.array(intersections[0].to_global()))
scene.make_figure()

## Sampling the Reachable Volume

Refraction through the duct wall means the relationship between stage
position and crossing point is nonlinear -- there is no closed-form inverse.
The approach used throughout this framework is the same two-step strategy
used for the flat-plate case, just applied to a harder geometry:

1. **Sample**: move the stage over a small neighborhood around our
   approximate starting position and record the *actual* (refracted)
   crossing point for each one.
2. **Fit**: train a fast regressor on those samples that maps a desired
   crossing point back to the stage position that produced it.

That regressor then supplies the *initial guess* for `Optimizer`, which
refines it with a gradient-free numerical search to micrometer precision.

In [ ]:
half_range = 4e-2  # sample within +-8 mm of the starting stage position
n_samples = 200

original_origin = source.origin.copy()
rng = np.random.default_rng(seed=0)

sampled_targets = []
sampled_origins = []

for _ in (
    pbar := tqdm(range(n_samples), desc="sampling", postfix={"intersections": 0})
):
    offset = rng.uniform(-half_range, half_range, size=3)
    candidate_origin = original_origin + stage_frame.vector(*offset)
    source.set_origin(candidate_origin)

    _, intersections = tracer.trace_and_find_crossings(sources=[source], threshold=1e-3)

    if len(intersections) == 1:
        # both target and origin are stored in stage_frame-local coordinates,
        # matching what GradientBoostingEstimator expects
        sampled_targets.append(np.array(intersections[0].to_frame(stage_frame)))
        sampled_origins.append(np.array(candidate_origin.to_frame(stage_frame)))

    pbar.set_postfix({"intersections": len(sampled_targets)})

source.set_origin(original_origin)

X = np.array(sampled_targets)
y = np.array(sampled_origins)
print(f"{len(X)}/{n_samples} samples produced exactly one crossing")

Since the tracked positions are stage displacements rather than plain points, add them to the scene as pairs: each sampled crossing point alongside the stage origin that produced it, both converted back to world coordinates.

In [ ]:
scene.clear_points()

for point, position in zip(X, y):
    scene.add_point(*stage_frame.point(point).to_global())
    scene.add_point(*stage_frame.point(position).to_global())

scene.make_figure()

In [ ]:
crossings_world = np.array([stage_frame.point(*point).to_global() for point in X])

fig = plt.figure(figsize=(5, 5))
ax3d = fig.add_subplot(projection="3d")
ax3d.scatter(*crossings_world.T, s=8)
ax3d.set_xlabel("X (m)")
ax3d.set_ylabel("Y (m)")
ax3d.set_zlabel("Z (m)")
ax3d.set_title("Sampled crossing points (world frame)")
plt.tight_layout()

## Training an Estimator and Building the Optimizer

`GradientBoostingEstimator` is fit once on the sampled data. `Optimizer`
then combines it with `RayTracer` to solve the actual inverse problem for
any target point: the estimator supplies a fast initial guess, and
Nelder-Mead refines it against the true (refracted) ray-traced crossing.

In [ ]:
estimator = lcc.optimization.GradientBoostingEstimator(frame=stage_frame).fit(X=X, y=y)
optimizer = lcc.optimization.Optimizer(tracer=tracer, estimator=estimator)

# a target roughly in the middle of the sampled region
target_point = root.point(0.05, -0.1, 0.0)
result = optimizer.find_source_origin(target=target_point, source=source)

print("converged:", result.success, " final error (m):", result.fun)

source.set_origin(result.x)
_, verify_intersections = tracer.trace_and_find_crossings(
    sources=[source], threshold=1e-3
)
print("requested target: ", np.array(target_point.to_global()))
print("achieved crossing:", np.array(verify_intersections[0].to_global()))
source.set_origin(original_origin)

## Building a Calibration Grid

With the optimizer working, we can now request a small regular grid of
target points and let it compute the stage position for each one. Points are
visited in a snake (boustrophaustrophedon) order -- filling each XY layer
before moving to the next Z layer -- so the stage never has to jump back
across the grid between neighboring points, minimizing travel time.

In [ ]:
def generate_snake_path(
    x_coords: np.ndarray, y_coords: np.ndarray, z_coords: np.ndarray
) -> np.ndarray:
    """Order a regular (x, y, z) grid to minimize travel between points.

    Fills each XY layer in a back-and-forth (snake) pattern before moving
    to the next Z layer, alternating the starting direction of each row so
    consecutive points are always adjacent.
    """
    path = []
    for iz, z in enumerate(z_coords):
        y_range = y_coords if iz % 2 == 0 else y_coords[::-1]
        for iy, y in enumerate(y_range):
            x_range = x_coords if (iz + iy) % 2 == 0 else x_coords[::-1]
            for x in x_range:
                path.append((x, y, z))
    return np.array(path)

In [ ]:
# a small grid centered on the sampled/trained region, well within its bounds
grid_x = np.linspace(0.053, 0.067, 3)
grid_y = np.linspace(-0.165, -0.153, 2)
grid_z = np.linspace(-0.006, 0.006, 2)

grid_points_world = generate_snake_path(grid_x, grid_y, grid_z)
print(f"{len(grid_points_world)} calibration points")

In [ ]:
stage_positions = []
achieved_crossings = []

for point in tqdm(grid_points_world, desc="optimizing grid"):
    target = root.point(*point)
    result = optimizer.find_source_origin(target=target, source=source)
    stage_positions.append(result.x)

    source.set_origin(result.x)
    _, crossing = tracer.trace_and_find_crossings(sources=[source], threshold=1e-4)
    achieved_crossings.append(
        np.array(crossing[0].to_global()) if crossing else np.ones(3) * np.nan
    )
    source.set_origin(original_origin)

errors = np.linalg.norm(np.array(achieved_crossings) - grid_points_world, axis=1)
print(f"max deviation from requested target: {errors.max() * 1e6:.2f} um")

In [ ]:
stage_positions_world = np.array([pos.to_global() for pos in stage_positions])

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].plot(*grid_points_world[:, [0, 1]].T, "o-", markersize=4)
axs[0].set_xlabel("X (m)")
axs[0].set_ylabel("Y (m)")
axs[0].set_title("Calibration grid (snake order)")
axs[0].set_aspect("equal")

axs[1].plot(*stage_positions_world[:, [0, 1]].T, "o-", markersize=4, color="tab:orange")
axs[1].set_xlabel("X (m)")
axs[1].set_ylabel("Y (m)")
axs[1].set_title("Corresponding stage positions")
axs[1].set_aspect("equal")
plt.tight_layout()

## Exporting G-code

The library does not (yet -- see the project README) generate G-code
directly from `Optimizer` results. Bridging the two is a small, manual step:
convert the simulated stage positions (relative, in meters) into the
controller's absolute machine coordinates (millimeters), then wrap each move
in whatever dwell/trigger sequence your camera needs.

The conversion needs exactly one physically measured reference point: where
your simulated `stage_frame` origin actually sits in real machine
coordinates. `basis_real` below stands in for that measurement -- on your
own setup you would jog the real stage to the position corresponding to
`stage_frame`'s origin, read its machine coordinates off the controller, and
use that instead. It is a property of the stage/controller mapping itself,
not of this specific duct, but it **will** drift if the stage is remounted,
re-homed differently, or the controller is replaced, so re-measure it rather
than reusing an old value indefinitely.

The trigger sequence (`M42 P67 ...`) toggles a camera-trigger pin over
Marlin's raw-pin G-code. Adjust the pin number (and command) to match your
own controller wiring.

In [ ]:
# stand-in for a physically measured reference point: on a real setup,
# jog the stage to stage_frame's origin and read these off the controller
basis_real_mm = np.array([27.8, 23.5, 29.5])

# stage positions relative to the simulated origin, in millimeters
relative_positions_mm = (
    stage_positions_world - np.array(original_origin.to_global())
) * 1000

dwell_ms = 500
trigger_sequence = "G4 P250\nM42 P67 S255\nG4 P10\nM42 P67 S0\nG4 P2\n"

gcode_lines = ["G90", ""]  # absolute positioning
gcode_lines.append(
    f"G1 X{basis_real_mm[0]:.3f} Y{basis_real_mm[1]:.3f} Z{basis_real_mm[2]:.3f}"
)
gcode_lines.append(f"G4 P{dwell_ms}")
gcode_lines.append("")

for offset in relative_positions_mm:
    coordinate = basis_real_mm + offset
    gcode_lines.append(
        f"G1 X{coordinate[0]:.3f} Y{coordinate[1]:.3f} Z{coordinate[2]:.3f}"
    )
    gcode_lines.append(f"G4 P{dwell_ms}")
    gcode_lines.append(trigger_sequence.strip("\n"))
    gcode_lines.append("")

gcode = "\n".join(gcode_lines)
print(gcode[:500] + "\n...")

Save `gcode` to a `.gcode` file and it is ready to run through
`lcc-control-gui`'s `StageController.run_gcode_file`, or via the GUI's
**Run GCode** button (see `lcc-control-gui/README.md`).

## Summary and Next Steps

This tutorial covered the parts of the workflow that go beyond the flat-plate
introduction:

1. **STL duct geometry**: loading STL meshes for multiple nested interfaces,
   and why normal orientation matters even more once surfaces are not flat.
2. **Stage placement**: separating configurable rig properties (mount angle,
   arm lengths) from the placement of whatever part is currently being
   measured.
3. **Sampling + estimation**: why the stage-position-to-crossing-point
   relationship needs to be learned rather than inverted analytically once
   refraction is involved.
4. **`Optimizer`**: refining an estimator's guess to micrometer precision
   for an arbitrary target point.
5. **Calibration grids**: generating a travel-minimizing scan order and
   solving the whole grid automatically.
6. **G-code export**: the current manual bridge between simulated stage
   positions and real machine coordinates.

From here, natural extensions are: sampling a larger neighborhood for a
larger calibration volume, swapping in your own STL geometry (checking
normal orientation first), or writing a small helper around the G-code
section above if you find yourself repeating it often -- that helper is
exactly the "planned `gcode` subpackage" mentioned in the project README.